In [3]:
# Initializing the project and the keys
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
import json
import traceback

# Update this line to specify your project id
project_id = "5f00e1ce-d082-4a2b-b08f-889658e932b7"

fablib = fablib_manager(project_id=project_id)

#fablib.show_config()

fablib.verify_and_configure()
!chmod 0600 fabric_config/fabric_bastion_key
!chmod 0600 fabric_config/slice_key



User: dbm4@alfred.edu bastion key is valid!
Configuration is valid
User: dbm4@alfred.edu bastion key is valid!
Configuration is valid
Please save the config!


In [4]:
slice_name = 'My_P4Kube_slice'
site_name = 'MICH'

num_workers = 6  # Tofino gives 8 ports: master + client + 6 workers

try:
    slice = fablib.get_slice(slice_name)
    slice.list_nodes()
    print('Done retrieving existing slice')

except Exception as e:
    print(f'Creating new slice: {slice_name}')
    print(f'Previous error: {e}')

    slice = fablib.new_slice(name=slice_name)

    vm_names = ['master', 'client'] + [f'node{i}' for i in range(1, num_workers + 1)]

    for vm_name in vm_names:
        node = slice.add_node(
            name=vm_name,
            site=site_name,
            image='default_ubuntu_20'
        )
        node.set_capacities(cores=2, ram=6, disk=10)
        node.add_component(model='NIC_Basic', name='if1')

    p4switch = slice.add_switch(
        name='p4switch',
        site=site_name
    )

    sw_ifaces = p4switch.get_interfaces()

    print("Tofino switch interfaces:")
    for idx, iface in enumerate(sw_ifaces):
        print(idx, iface.get_name())

    needed_ports = 2 + num_workers
    if len(sw_ifaces) < needed_ports:
        raise Exception(f"Need {needed_ports} switch interfaces, found {len(sw_ifaces)}")

    links = {}

    links['ms'] = slice.add_l2network(name='ms')
    links['ms'].add_interface(slice.get_node('master').get_interface('master-if1-p1'))
    links['ms'].add_interface(sw_ifaces[0])

    links['cs'] = slice.add_l2network(name='cs')
    links['cs'].add_interface(slice.get_node('client').get_interface('client-if1-p1'))
    links['cs'].add_interface(sw_ifaces[1])

    for i in range(1, num_workers + 1):
        link_name = f'n{i}s'
        links[link_name] = slice.add_l2network(name=link_name)

        host = slice.get_node(f'node{i}')
        links[link_name].add_interface(host.get_interface(f'node{i}-if1-p1'))
        links[link_name].add_interface(sw_ifaces[i + 1])

    slice.submit()
    print(f"Slice '{slice_name}' created and submitted.")

ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
2d1dadbf-63f1-4103-93ce-c9ed0d3d8d73,client,2,8,10,default_ubuntu_20,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,2607:f018:110:11:f816:3eff:fe9f:308a,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fe9f:308a,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
455d32e6-f07a-4d7a-8bcd-179f91298877,master,2,8,10,default_ubuntu_20,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,2607:f018:110:11:f816:3eff:fe92:2843,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fe92:2843,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
dde8b80e-e03b-4fbc-a516-aafb4fc228e9,node1,2,8,10,default_ubuntu_20,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,2607:f018:110:11:f816:3eff:feb4:e2d5,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:feb4:e2d5,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
e546cc2e-2273-45e3-add2-3292a8125c74,node2,2,8,10,default_ubuntu_20,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,2607:f018:110:11:f816:3eff:fe6b:965b,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fe6b:965b,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
5d3ac195-f9a0-49ff-b34f-2f7763f5dcf2,node3,2,8,10,default_ubuntu_20,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,2607:f018:110:11:f816:3eff:fe44:e2b9,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fe44:e2b9,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
ba156b8a-c73c-43ab-9263-366d118e4ba1,node4,2,8,10,default_ubuntu_20,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,2607:f018:110:11:f816:3eff:fe58:14f0,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fe58:14f0,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
d9047c4a-a83b-4e33-aed2-04ba0bdb0943,node5,2,8,10,default_ubuntu_20,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,2607:f018:110:11:f816:3eff:fe11:d826,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fe11:d826,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
4e2b7c60-db39-460c-8b87-a5bda7bf38c8,node6,2,8,10,default_ubuntu_20,qcow2,mich-w2.fabric-testbed.net,MICH,ubuntu,2607:f018:110:11:f816:3eff:fe4b:8fdb,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fe4b:8fdb,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
4276f864-2f66-452e-9112-4ee3d6786928,p4switch,,,,,,,MICH,fabric,2607:f018:110:11:290:fbff:fe76:d00b,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config fabric@2607:f018:110:11:290:fbff:fe76:d00b,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


Done retrieving existing slice


In [7]:
print('Changing the MAC addresses of all nodes, master and client')

mac_map = {
    'master': '5E:00:00:00:00:01',
    'client': '5E:00:00:00:00:02',
    'node1':  '5E:00:00:00:00:03',
    'node2':  '5E:00:00:00:00:04',
    'node3':  '5E:00:00:00:00:05',
    'node4':  '5E:00:00:00:00:06',
    'node5':  '5E:00:00:00:00:07',
    'node6':  '5E:00:00:00:00:08',
}

ip_map = {
    'master': '10.0.0.2/24',
    'client': '11.0.0.2/24',
    'node1':  '12.0.0.2/24',
    'node2':  '13.0.0.2/24',
    'node3':  '14.0.0.2/24',
    'node4':  '15.0.0.2/24',
    'node5':  '16.0.0.2/24',
    'node6':  '17.0.0.2/24',
}

gw_map = {
    'master': '10.0.0.1',
    'client': '11.0.0.1',
    'node1':  '12.0.0.1',
    'node2':  '13.0.0.1',
    'node3':  '14.0.0.1',
    'node4':  '15.0.0.1',
    'node5':  '16.0.0.1',
    'node6':  '17.0.0.1',
}

for node_name in ['master', 'client'] + [f'node{i}' for i in range(1, 7)]:
    node = slice.get_node(node_name)
    iface = node.get_interface(f"{node_name}-if1-p1").get_os_interface()

    config_script = f"""
        sudo apt-get update
        sudo apt-get install -y net-tools
        sudo ip link set dev {iface} down
        sudo ip link set dev {iface} address {mac_map[node_name]}
        sudo ip addr flush dev {iface}
        sudo ip addr add {ip_map[node_name]} dev {iface}
        sudo ip link set dev {iface} up
        sudo ip route replace default via {gw_map[node_name]}
    """

    node.execute(config_script)

print('Done, move to the next step')

Changing the MAC addresses of all nodes, master and client
Get:1 http://security.ubuntu.com/ubuntu focal-security InRelease [128 kB]
Hit:2 http://nova.clouds.archive.ubuntu.com/ubuntu focal InRelease
Get:3 http://security.ubuntu.com/ubuntu focal-security/main amd64 Packages [3610 kB]
Get:4 http://nova.clouds.archive.ubuntu.com/ubuntu focal-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu focal-security/main Translation-en [518 kB]
Get:6 http://security.ubuntu.com/ubuntu focal-security/restricted amd64 Packages [3828 kB]
Get:7 http://security.ubuntu.com/ubuntu focal-security/universe amd64 Packages [1052 kB]
Get:8 http://security.ubuntu.com/ubuntu focal-security/universe Translation-en [220 kB]
Get:9 http://security.ubuntu.com/ubuntu focal-security/universe amd64 c-n-f Metadata [22.4 kB]
Get:10 http://security.ubuntu.com/ubuntu focal-security/multiverse amd64 Packages [26.7 kB]
Get:11 http://security.ubuntu.com/ubuntu focal-security/multiverse Translation-en [6416 B]
G

In [8]:
print('Installing docker, Kubernetes on master and all nodes')
for node_name in ['master'] + [f'node{i}' for i in range(1, 7)]:
    node = slice.get_node(node_name)
    script = """
        # Docker install
        sudo apt update;
        sudo apt upgrade -y;
        sudo apt install docker.io -y;
        sudo systemctl enable docker;
        sudo systemctl start docker;

        # Kubernetes install
        cd /etc/apt && sudo mkdir -p keyrings && cd ~;
        echo "deb [signed-by=/etc/apt/keyrings/kubernetes-apt-keyring.gpg] https://pkgs.k8s.io/core:/stable:/v1.28/deb/ /" | sudo tee /etc/apt/sources.list.d/kubernetes.list;
        curl -fsSL https://pkgs.k8s.io/core:/stable:/v1.28/deb/Release.key | sudo gpg --dearmor -o /etc/apt/keyrings/kubernetes-apt-keyring.gpg;
        sudo apt-get update;
        sudo apt-get install -y kubelet kubeadm kubectl;
        sudo apt-mark hold kubeadm kubelet kubectl;
    """
    node.execute(script)
print('All done, move to the next step')

Installing docker, Kubernetes on master and all nodes


Hit:1 http://security.ubuntu.com/ubuntu focal-security InRelease
Hit:2 http://nova.clouds.archive.ubuntu.com/ubuntu focal InRelease
Hit:3 http://nova.clouds.archive.ubuntu.com/ubuntu focal-updates InRelease
Hit:4 http://nova.clouds.archive.ubuntu.com/ubuntu focal-backports InRelease
Reading package lists...
Building dependency tree...
Reading state information...
21 packages can be upgraded. Run 'apt list --upgradable' to see them.


Reading package lists...
Building dependency tree...
Reading state information...
Calculating upgrade...
The following security updates require Ubuntu Pro with 'esm-infra' enabled:
  fdisk bind9-dnsutils cloud-init libnghttp2-14 libpackagekit-glib2-18
  pollinate uuid-runtime linux-headers-generic libfdisk1 gnupg-utils
  vim-common libcurl4 gpg-wks-client openssl libblockdev-swap2 gnupg-l10n
  libssh-4 libpython3.8-minimal git-man python3-jwt libsystemd0 gcc-10-base
  libkmod2 libmount1 snapd libsqlit

In [9]:
print('Setting up the cluster')
node = slice.get_node('master')
script = 'sudo kubeadm init --pod-network-cidr=10.244.0.0/16'
node.execute(script)
print('Update the token in the next cell based on the output above')

Setting up the cluster
I0526 15:31:56.886280    7165 version.go:256] remote version is much newer: v1.36.1; falling back to: stable-1.28
[init] Using Kubernetes version: v1.28.15
[preflight] Running pre-flight checks
[preflight] Pulling images required for setting up a Kubernetes cluster
[preflight] This might take a minute or two, depending on the speed of your internet connection
[preflight] You can also perform this action in beforehand using 'kubeadm config images pull'
W0526 15:32:04.641116    7165 checks.go:835] detected that the sandbox image "registry.k8s.io/pause:3.8" of the container runtime is inconsistent with that used by kubeadm. It is recommended that using "registry.k8s.io/pause:3.9" as the CRI sandbox image.
[certs] Using certificateDir folder "/etc/kubernetes/pki"
[certs] Generating "ca" certificate and key
[certs] Generating "apiserver" certificate and key
[certs] apiserver serving cert is signed for DNS names [kubernetes kubernetes.default kubernetes.default.svc kub

In [1]:
JOIN_COMMAND = 'sudo ' + '''kubeadm join 10.0.0.2:6443 --token qc6gzt.xt65aie8576g22w3 \
	--discovery-token-ca-cert-hash sha256:1cf5ec5084cf4b87077fcce50791454deedf95e1821cef6904792e3c655e4fcb'''
print(JOIN_COMMAND)

sudo kubeadm join 10.0.0.2:6443 --token qc6gzt.xt65aie8576g22w3 	--discovery-token-ca-cert-hash sha256:1cf5ec5084cf4b87077fcce50791454deedf95e1821cef6904792e3c655e4fcb


In [5]:
print('Joining all worker nodes to the cluster')
for node_name in [f'node{i}' for i in range(1, 7)]:
    node = slice.get_node(node_name)
    node.execute(JOIN_COMMAND)
print('All done, move to the next step')

Joining all worker nodes to the cluster
[preflight] Running pre-flight checks
error execution phase preflight: couldn't validate the identity of the API Server: Get "https://10.0.0.2:6443/api/v1/namespaces/kube-public/configmaps/cluster-info?timeout=10s": dial tcp 10.0.0.2:6443: connect: no route to host
To see the stack trace of this error execute with --v=5 or higher
[preflight] Running pre-flight checks


KeyboardInterrupt: 

In [12]:
print('Configuring networking for Kubernetes via Calico CNI')

node = slice.get_node('master')

script = '''
sudo mkdir -p $HOME/.kube

sudo cp -i /etc/kubernetes/admin.conf $HOME/.kube/config

sudo chown $(id -u):$(id -g) $HOME/.kube/config

kubectl apply -f https://docs.projectcalico.org/manifests/calico.yaml
'''

node.execute(script)

print('Done on the master node')

# Fix kubelet node-ip configuration
all_nodes = ['master'] + [f'node{i}' for i in range(1, 7)]

node_ip_map = {
    'master': '10.0.0.2',
    'node1': '12.0.0.2',
    'node2': '13.0.0.2',
    'node3': '14.0.0.2',
    'node4': '15.0.0.2',
    'node5': '16.0.0.2',
    'node6': '17.0.0.2',
}

for node_name in all_nodes:
    node = slice.get_node(node_name)
    ip = node_ip_map[node_name]

    script = f'''
echo "KUBELET_EXTRA_ARGS=--node-ip={ip}" | sudo tee /etc/default/kubelet

sudo systemctl daemon-reexec
sudo systemctl daemon-reload
sudo systemctl restart kubelet
'''

    print(f"Configuring kubelet on {node_name}")
    node.execute(script)

print('All done, move to the next step')

Configuring networking for Kubernetes via Calico CNI
poddisruptionbudget.policy/calico-kube-controllers created
serviceaccount/calico-kube-controllers created
serviceaccount/calico-node created
configmap/calico-config created
customresourcedefinition.apiextensions.k8s.io/bgpconfigurations.crd.projectcalico.org created
customresourcedefinition.apiextensions.k8s.io/bgppeers.crd.projectcalico.org created
customresourcedefinition.apiextensions.k8s.io/blockaffinities.crd.projectcalico.org created
customresourcedefinition.apiextensions.k8s.io/caliconodestatuses.crd.projectcalico.org created
customresourcedefinition.apiextensions.k8s.io/clusterinformations.crd.projectcalico.org created
customresourcedefinition.apiextensions.k8s.io/felixconfigurations.crd.projectcalico.org created
customresourcedefinition.apiextensions.k8s.io/globalnetworkpolicies.crd.projectcalico.org created
customresourcedefinition.apiextensions.k8s.io/globalnetworksets.crd.projectcalico.org created
customresourcedefinition

In [13]:
print('Cloning P4Kube repo on the master node')
node = slice.get_node('master')
script = f'git clone https://github.com/gareging/P4Kube.git;'
node.execute(script)
print('All done, move to the next step')

Cloning P4Kube repo on the master node
Cloning into 'P4Kube'...
All done, move to the next step


In [14]:
# Setup the metrics and P4Kube plugin
node = slice.get_node('master')
script = '''
    wget -c https://go.dev/dl/go1.22.0.linux-amd64.tar.gz;
    sudo tar -C /usr/local/ -xzf go1.22.0.linux-amd64.tar.gz;
    export PATH=$PATH:/usr/local/go/bin;
    cd ~/P4Kube/
    kubectl apply -f metrics-server.yaml;
    kubectl patch deployment metrics-server -n kube-system --type 'json' -p '[{"op": "add", "path": "/spec/template/spec/containers/0/args/-", "value": "--kubelet-insecure-tls"}]';
    cd ~/P4Kube/kubectl-plugin;
    go get k8s.io/client-go@latest;
    go get k8s.io/metrics/pkg/client/clientset/versioned;
    go get k8s.io/apimachinery/pkg/util/diff@v0.34.0;
    cp main.go_LOAD_AWARE main.go;
    echo 'export PATH="/home/ubuntu/P4Kube/kubectl-plugin/:$PATH"' >> ~/.bashrc;
    go build;
'''
node.execute(script)

print('All done')

--2026-05-26 16:06:26--  https://go.dev/dl/go1.22.0.linux-amd64.tar.gz
Resolving go.dev (go.dev)... 2001:4860:4802:38::15, 2001:4860:4802:34::15, 2001:4860:4802:36::15, ...
Connecting to go.dev (go.dev)|2001:4860:4802:38::15|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://dl.google.com/go/go1.22.0.linux-amd64.tar.gz [following]
--2026-05-26 16:06:26--  https://dl.google.com/go/go1.22.0.linux-amd64.tar.gz
Resolving dl.google.com (dl.google.com)... 2607:f8b0:4009:810::200e, 142.250.217.110
Connecting to dl.google.com (dl.google.com)|2607:f8b0:4009:810::200e|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 68988925 (66M) [application/x-gzip]
Saving to: ‘go1.22.0.linux-amd64.tar.gz’

     0K .......... .......... .......... .......... ..........  0% 3.49M 19s
    50K .......... .......... .......... .......... ..........  0% 5.89M 15s
   100K .......... .......... .......... .......... ..........  0% 8.55M 13s
   150K ........

In [15]:
print("Find the master ssh command below:", slice.get_node(name="master"))
print('ssh into master, open tmux and run the command shown below')
print()
command='''cd ~/P4Kube/kubectl-plugin; export PATH=$PATH:/usr/local/go/bin; echo 'export PATH="/home/ubuntu/P4Kube/kubectl-plugin/:$PATH"' >> ~/.bashrc; go build; kubectl p4-loadbalancer;'''
print(command)

Find the master ssh command below: -----------------  ------------------------------------------------------------------------------------------------------------------------------------------
ID                 455d32e6-f07a-4d7a-8bcd-179f91298877
Name               master
Cores              2
RAM                8
Disk               10
Image              default_ubuntu_20
Image Type         qcow2
Host               mich-w2.fabric-testbed.net
Site               MICH
Management IP      2607:f018:110:11:f816:3eff:fe92:2843
Reservation State  Active
Error Message
SSH Command        ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2607:f018:110:11:f816:3eff:fe92:2843
-----------------  ------------------------------------------------------------------------------------------------------------------------------------------
ssh into master, open tmux and run the command shown below

cd ~/P4Kube/kubectl-plugin; export PATH=$PATH:/usr/local/

In [19]:
# Setup the deployment and service (optional)
print('Installing nginx deployment with 10 replicas')
node = slice.get_node('master')
script = '''     
    cd P4Kube/;
    kubectl create -f nginx-deployment.yaml;
    kubectl create -f nginx-service.yaml;
    kubectl patch svc ngnix-service -p '{"spec":{"externalTrafficPolicy":"Local"}}';
'''
node.execute(script)
print('All done, move to the next step')

Installing nginx deployment with 10 replicas
deployment.apps/nginx created
service/ngnix-service created
service/ngnix-service patched
All done, move to the next step


In [20]:
print('On the P4Switch node, simple_switch_CLI will show 10 for the register replica_count (1 deployment)')
print('''To check, run 'register_read replica_count' in simple_switch_CLI''')
print("Find the switch ssh command below:", slice.get_node(name="p4switch"))

On the P4Switch node, simple_switch_CLI will show 10 for the register replica_count (1 deployment)
To check, run 'register_read replica_count' in simple_switch_CLI
Find the switch ssh command below: -----------------  -----------------------------------------------------------------------------------------------------------------------------------------
ID                 08aafd3e-f732-495a-a360-d8d7a1d27b45
Name               p4switch
Cores              2
RAM                8
Disk               10
Image              default_ubuntu_20
Image Type         qcow2
Host               rutg-w1.fabric-testbed.net
Site               RUTG
Management IP      2620:0:d61:4101:f816:3eff:fe39:f6a5
Reservation State  Active
Error Message
SSH Command        ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:fe39:f6a5
-----------------  ---------------------------------------------------------------------------------------------

In [21]:
'''
This code will automatically replace the default html with a string "nodeX" to observe load balancing 
'''
for node_name in [f'node{i}' for i in range(1, 11)]:
    node = slice.get_node(node_name)
    script = '''
        podid=$(sudo crictl pods | awk '$0 ~ /nginx/ {print $1}');
        dockerid=$(sudo crictl ps | grep $podid | awk '{print $1}');
    '''
    script += f'''echo {node_name} | sudo crictl exec -i "$dockerid" sh -c 'cat > /usr/share/nginx/html/index.html';'''
    node.execute(script)
    print('Done on node', node_name)

time="2025-11-17T22:40:26Z" level=warning msg="runtime connect using default endpoints: [unix:///var/run/dockershim.sock unix:///run/containerd/containerd.sock unix:///run/crio/crio.sock unix:///var/run/cri-dockerd.sock]. As the default settings are now deprecated, you should set the endpoint instead."
time="2025-11-17T22:40:26Z" level=error msg="validate service connection: validate CRI v1 runtime API for endpoint \"unix:///var/run/dockershim.sock\": rpc error: code = Unavailable desc = connection error: desc = \"transport: Error while dialing: dial unix /var/run/dockershim.sock: connect: no such file or directory\""
time="2025-11-17T22:40:26Z" level=warning msg="runtime connect using default endpoints: [unix:///var/run/dockershim.sock unix:///run/containerd/containerd.sock unix:///run/crio/crio.sock unix:///var/run/cri-dockerd.sock]. As the default settings are now deprecated, you should set the endpoint instead."
time="2025-11-17T22:40:26Z" level=error msg="validate service connecti

In [22]:
print('Login to client and run curl 10.0.0.2 to verify load balancing')
print("Find the client ssh command below:", slice.get_node(name="client"))

Login to client and run curl 10.0.0.2 to verify load balancing
Find the client ssh command below: -----------------  -----------------------------------------------------------------------------------------------------------------------------------------
ID                 d2fea52e-04c8-4ab6-bd59-13a1d609c8ed
Name               client
Cores              2
RAM                8
Disk               10
Image              default_ubuntu_20
Image Type         qcow2
Host               rutg-w1.fabric-testbed.net
Site               RUTG
Management IP      2620:0:d61:4101:f816:3eff:feab:5b76
Reservation State  Active
Error Message
SSH Command        ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:feab:5b76
-----------------  -----------------------------------------------------------------------------------------------------------------------------------------


In [ ]:
print('Downscale the first deployment to 7 nodes')
node = slice.get_node('master')
script = 'kubectl scale --replicas=7 deployment nginx'
node.execute(script)
print('Check replica_count register on p4switch')

In [23]:
print('Install second nginx deployment with 3 replicas')
node = slice.get_node('master')
script = '''     
    cd P4Kube/;
    kubectl create -f nginx-deployment2.yaml;
    kubectl create -f nginx-service2.yaml;
    kubectl patch svc ngnix-service2 -p '{"spec":{"externalTrafficPolicy":"Local"}}';
'''
node.execute(script)
print('On the P4Switch node, replica_count register will show 7, 3 (1st and 2nd deployment)')
print('''To check, run 'register_read replica_count' in simple_switch_CLI''')
print(''' To test from the client, run curl 10.0.0.3 ''')

Install second nginx deployment with 3 replicas
deployment.apps/nginx2 created
service/ngnix-service2 created
service/ngnix-service2 patched
On the P4Switch node, replica_count register will show 7, 3 (1st and 2nd deployment)
To check, run 'register_read replica_count' in simple_switch_CLI
 To test from the client, run curl 10.0.0.3 


In [28]:
print('Install AIOQUIC server as the third deployment')
node = slice.get_node('master')
script = '''     
    cd P4Kube/;
    kubectl create -f quic.yaml;
    kubectl patch svc aioquic-http3-service -p '{"spec":{"externalTrafficPolicy":"Local"}}';
'''
node.execute(script)
print('All done')

Install AIOQUIC server as the third deployment
deployment.apps/aioquic-http3 created
service/aioquic-http3-service created
service/aioquic-http3-service patched
All done


In [25]:
# Testing QUIC (part 1)
print('Install QUIC dependencies on the client')
node = slice.get_node('client')
script = '''     
    sudo apt install libssl-dev python3.9 python3.9-venv python3.9-dev -y;
    sudo apt install python3-pip -y;
    pip3 install aioquic wsproto;
'''
node.execute(script)
print('All done')

Install QUIC dependencies on the client
Reading package lists...


Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libc-dev-bin libc6-dev libcrypt-dev libexpat1-dev libpython3.9
  libpython3.9-dev libpython3.9-minimal libpython3.9-stdlib linux-libc-dev
  manpages-dev python-pip-whl python3.9-minimal zlib1g-dev
Suggested packages:
  glibc-doc libssl-doc python3.9-doc binutils binfmt-support
The following NEW packages will be installed:
  libc-dev-bin libc6-dev libcrypt-dev libexpat1-dev libpython3.9
  libpython3.9-dev libpython3.9-minimal libpython3.9-stdlib libssl-dev
  linux-libc-dev manpages-dev python-pip-whl python3.9 python3.9-dev
  python3.9-minimal python3.9-venv zlib1g-dev
0 upgraded, 17 newly installed, 0 to remove and 18 not upgraded.
Need to get 21.1 MB of archives.
After this operation, 89.8 MB of additional disk space will be used.
Get:1 http://nova.clouds.archive.ubuntu.com/ubuntu focal-updates/universe amd64

In [26]:
# Testing QUIC (part 2)
print('Clone AIOQUIC repo')
node = slice.get_node('client')
script = '''     
    git clone https://github.com/aiortc/aioquic.git;
'''
node.execute(script)
print('All done')

Clone AIOQUIC repo
Cloning into 'aioquic'...
All done


In [27]:
# Testing QUIC (part 3)
print('Run QUIC request')
node = slice.get_node('client')
script = '''     
    cd aioquic/examples;
    python3 http3_client.py --insecure https://10.0.0.4:4433/;
'''
node.execute(script)

Run QUIC request
Traceback (most recent call last):
  File "http3_client.py", line 124, in <module>
    class HttpClient(QuicConnectionProtocol):
  File "http3_client.py", line 159, in HttpClient
    self, url: str, subprotocols: Optional[list[str]] = None
TypeError: 'type' object is not subscriptable


('',
 'Traceback (most recent call last):\n  File "http3_client.py", line 124, in <module>\n    class HttpClient(QuicConnectionProtocol):\n  File "http3_client.py", line 159, in HttpClient\n    self, url: str, subprotocols: Optional[list[str]] = None\nTypeError: \'type\' object is not subscriptable\n')